In [14]:
import pandas as pd 
import numpy as np 

returns = pd.read_parquet("../../data/01_raw/returns.parquet")
nan_ratio = returns.isna().mean()
returns = returns.loc[:, nan_ratio <= 0.30]

returns.isna().sum(axis=0).sum()

np.int64(30008)

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.kalman_filter import KalmanFilter
from statsmodels.tsa.statespace.mlemodel import MLEModel

# Carrega os dados
returns = pd.read_parquet("../../data/01_raw/returns.parquet")
nan_ratio = returns.isna().mean()
returns = returns.loc[:, nan_ratio <= 0.30]

# Função para imputar uma série temporal usando Kalman Filter
def kalman_impute(series):
    class SimpleKF(MLEModel):
        def __init__(self, endog):
            # endog é a série temporal
            super(SimpleKF, self).__init__(endog, k_states=1, initialization='approximate_diffuse')
            self['design'] = [1]
            self['transition'] = [1]
            self['selection'] = [1]
            self['state_cov'] = [1]
        
        def update(self, params, **kwargs):
            # Ajuste opcional de parâmetros (podemos manter como está)
            pass

    # Inicializa o modelo
    model = SimpleKF(series)
    # Aplica filtro de Kalman
    kf = model.smooth(series)
    # Retorna os valores imputados
    return pd.Series(kf.smoothed_state[0], index=series.index)

# Aplica coluna por coluna
returns_imputed = returns.apply(kalman_impute, axis=0)

returns_imputed


c:\Users\groque\Desktop\paper-financial-graph\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\groque\Desktop\paper-financial-graph\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\groque\Desktop\paper-financial-graph\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\groque\Desktop\paper-financial-graph\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but i